# MMPose vs MediaPipe Pose Comparison

Run this notebook on Google Colab with a T4 GPU runtime. It writes MMPose artifacts next to the existing MediaPipe pose artifacts and produces rule/classifier comparison tables.

### 1. Mount Drive

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Install MMPose

In [54]:
# Colab Python 3.12-friendly RTMW/RTMPose runtime. Avoid mmcv/mmdet native ops.
!pip uninstall -y -q openmim openxlab chumpy mmpose mmdet mmcv mmcv-lite mmengine xtcocotools
!pip install -q -U pip wheel "setuptools>=69"
!pip install -q -U rtmlib onnxruntime-gpu opencv-python numpy tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opendatalab 0.0.10 requires openxlab, which is not installed.
datasets 4.0.0 requires requests>=2.32.2, but you have requests 2.28.2 which is incompatible.
pymc 5.28.4 requires rich>=13.7.1, but you have rich 13.4.2 which is incompatible.
libpysal 4.14.1 requires requests>=2.32.0, but you have requests 2.28.2 which is incompatible.
pytensor 2.38.2 requires filelock>=3.15, but you have filelock 3.14.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
yfinance 0.2.66 requires requests>=2.31, but you have requests 2.28.2 which is incompatible.


In [55]:
import torch
import rtmlib
from rtmlib import Wholebody

print('cuda_available:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__)
print('rtmlib:', getattr(rtmlib, '__version__', 'unknown'))
print('rtmlib Wholebody import: OK')

cuda_available: True
gpu: Tesla T4
torch: 2.10.0+cu128
rtmlib: unknown
rtmlib Wholebody import: OK


### 3. Configure Paths

In [56]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path('/content/drive/MyDrive/x-coach')
DATASET_ROOT = PROJECT_ROOT / 'data' / 'Fitness-AQA_dataset_release' / 'Squat' / 'Labeled_Dataset'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'Squat' / 'Labeled_Dataset'
SPLIT_DIR = DATASET_ROOT / 'Splits'
LOCAL_VIDEO_DIR = Path('/content/local_videos')
LOCAL_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

video_zip = DATASET_ROOT / 'videos.zip'
if video_zip.exists():
    subprocess.run(['unzip', '-q', '-n', str(video_zip), '-d', str(LOCAL_VIDEO_DIR)], check=True)

VIDEO_ROOT = LOCAL_VIDEO_DIR if any(LOCAL_VIDEO_DIR.rglob('*.mp4')) else DATASET_ROOT / 'videos'
MMPOSE_JSON_DIR = OUTPUT_ROOT / 'mmpose_pose_json'
MMPOSE_FEATURE_DIR = OUTPUT_ROOT / 'mmpose_pose_features'
MMPOSE_VIEW_METADATA = OUTPUT_ROOT / 'mmpose_view_metadata.csv'
MMPOSE_RULE_DIR = OUTPUT_ROOT / 'mmpose_pose_rule_detections'
MMPOSE_RULE_SUMMARY = OUTPUT_ROOT / 'mmpose_pose_rule_detections_summary.csv'
MMPOSE_RULE_METRICS = OUTPUT_ROOT / 'mmpose_pose_rule_validation_metrics.csv'
MMPOSE_CLASSIFIER_ROOT = PROJECT_ROOT / 'data' / 'Squat' / 'mmpose_pose_classifier_experiments'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('VIDEO_ROOT:', VIDEO_ROOT)

PROJECT_ROOT: /content/drive/MyDrive/x-coach
DATASET_ROOT: /content/drive/MyDrive/x-coach/data/Fitness-AQA_dataset_release/Squat/Labeled_Dataset
OUTPUT_ROOT: /content/drive/MyDrive/x-coach/data/Squat/Labeled_Dataset
VIDEO_ROOT: /content/local_videos


### 4. Extract MMPose Whole-Body JSON

In [ ]:
# cmd = [
#     'python', 'src/run_mmpose_pose_extraction.py',
#     '--video-dir', str(VIDEO_ROOT),
#     '--split-dir', str(SPLIT_DIR),
#     '--output-dir', str(MMPOSE_JSON_DIR),
#     '--model', 'wholebody',
#     '--device', 'cuda:0',
# ]
# subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
!python "/content/drive/MyDrive/x-coach/src/run_mmpose_pose_extraction.py" \
  --video-dir "/content/local_videos" \
  --split-dir "/content/drive/MyDrive/x-coach/data/Fitness-AQA_dataset_release/Squat/Labeled_Dataset/Splits" \
  --output-dir "/content/drive/MyDrive/x-coach/data/Squat/Labeled_Dataset/mmpose_pose_json" \
  --model wholebody \
  --device cuda:0

Found 1623 videos to process from /content/local_videos.
Writing MMPose pose outputs under /content/drive/MyDrive/x-coach/data/Squat/Labeled_Dataset/mmpose_pose_json.
[1] Processing train/46777_1 from 46777_1.mp4...
Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/yolox_m_8xb8-300e_humanart-c2c7a14a.zip" to /root/.cache/rtmlib/hub/checkpoints/yolox_m_8xb8-300e_humanart-c2c7a14a.zip
100% 89.9M/89.9M [00:01<00:00, 66.2MB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
load /root/.cache/rtmlib/hub/checkpoints/yolox_m_8xb8-300e_humanart-c2c7a14a.onnx with onnxruntime backend
Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmw/onnx_sdk/rtmw-dw-x-l_simcc-cocktail14_270e-256x192_20231122.zip" to /root/.cache/rtmlib/hu

### 5. Convert MMPose JSON to Pose Features

In [ ]:
cmd = [
    'python', 'scripts/run_pose_feature_extraction.py',
    '--pose-json-dir', str(MMPOSE_JSON_DIR),
    '--split-dir', str(SPLIT_DIR),
    '--output-dir', str(MMPOSE_FEATURE_DIR),
    '--overwrite',
]
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

### 6. View Metadata and Rule Evaluation

In [ ]:
subprocess.run([
    'python', 'scripts/run_view_estimation.py',
    '--pose-json-dir', str(MMPOSE_JSON_DIR),
    '--split-dir', str(SPLIT_DIR),
    '--output', str(MMPOSE_VIEW_METADATA),
], cwd=PROJECT_ROOT, check=True)

subprocess.run([
    'python', 'scripts/run_pose_rule_detection.py',
    '--pose-json-dir', str(MMPOSE_JSON_DIR),
    '--split-dir', str(SPLIT_DIR),
    '--output-dir', str(MMPOSE_RULE_DIR),
    '--summary-output', str(MMPOSE_RULE_SUMMARY),
    '--no-retrieval',
], cwd=PROJECT_ROOT, check=True)

subprocess.run([
    'python', 'scripts/evaluate_pose_rule_detection.py',
    '--detections-dir', str(MMPOSE_RULE_DIR),
    '--view-metadata', str(MMPOSE_VIEW_METADATA),
    '--output', str(MMPOSE_RULE_METRICS),
], cwd=PROJECT_ROOT, check=True)

### 7. Train MMPose Pose-Only Classifiers

In [ ]:
subprocess.run([
    'python', 'scripts/run_videomae_experiment_grid.py',
    '--feature-dir', str(MMPOSE_FEATURE_DIR),
    '--train-keys', str(SPLIT_DIR / 'train_keys.json'),
    '--val-keys', str(SPLIT_DIR / 'val_keys.json'),
    '--test-keys', str(SPLIT_DIR / 'test_keys.json'),
    '--forward-labels', str(LABELED_ROOT / 'Labels' / 'error_knees_forward.json'),
    '--inward-labels', str(LABELED_ROOT / 'Labels' / 'error_knees_inward.json'),
    '--output-root', str(MMPOSE_CLASSIFIER_ROOT),
    '--label-modes', 'combined,knees_forward,knees_inward',
    '--seeds', '1,2,3,4,5',
    '--epochs', '20',
    '--lr', '3e-4',
    '--hidden-dim', '128',
    '--dropout', '0.4',
    '--weight-decay', '0.01',
    '--early-stopping-patience', '5',
    '--threshold-objective', 'balanced_accuracy',
    '--device', 'cuda',
    '--normalize-features',
], cwd=PROJECT_ROOT, check=True)

### 8. Write Backend Comparison Report

In [ ]:
subprocess.run([
    'python', 'scripts/compare_pose_backends.py',
    '--mmpose-pose-json-dir', str(MMPOSE_JSON_DIR),
    '--mmpose-rule-metrics', str(MMPOSE_RULE_METRICS),
    '--mmpose-classifier-summary', str(MMPOSE_CLASSIFIER_ROOT / 'metrics' / 'experiment_summary.csv'),
], cwd=PROJECT_ROOT, check=True)

comparison_md = PROJECT_ROOT / 'data' / 'Squat' / 'mmpose_mediapipe_comparison' / 'backend_comparison.md'
print(comparison_md.read_text()[:4000])